In [0]:
# ============================================================
# Cell 1 — Install
# ============================================================
%pip install sentence-transformers
dbutils.library.restartPython()
%pip install spacy

In [0]:
# ============================================================
# Cell 2 — Config & model
# ============================================================
from sentence_transformers import SentenceTransformer
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, FloatType, StringType, StructType, StructField
import pandas as pd
import numpy as np

CHECKPOINT_TABLE = "workspace.default.mcf_skills_course_embeddings_v1"

# --- Test revision configs ---
TEST_CONFIGS = {
    "TestA": {
        "name": "TestA",
        "chunk_size": 2000,
        "batch_size": 512,
        "description": "Baseline noun extraction for both course title and what you'll learn",
    },
    "TestB": {
        "name": "TestB",
        "chunk_size": 2000,
        "batch_size": 512,
        "description": "Noun extraction only descriptions, preserve course title",
    },
    "TestA3": {
        "name": "TestA3",
        "chunk_size": 2000,
        "batch_size": 512,
        "description": "Baseline noun + pronouns + adjectives extraction for both course title and what you'll learn",
    },
    "TestA2": {
        "name": "TestA2",
        "chunk_size": 300,
        "batch_size": 512,
        "description": "Noun extraction only descriptions, preserve course title, lower chunk size",
    },
    "TestC": {
        "name": "TestC",
        "chunk_size": 2000,
        "batch_size": 512,
        "description": "No noun extraction",
    },
    # Add more test configs as needed
}
CURRENT_TEST = "TestA3"
TEST_CONFIG = TEST_CONFIGS[CURRENT_TEST]

CHUNK_SIZE = TEST_CONFIG["chunk_size"]
BATCH_SIZE = TEST_CONFIG["batch_size"]

def reset_test_config(test_name):
    global CURRENT_TEST, TEST_CONFIG, CHUNK_SIZE, BATCH_SIZE
    CURRENT_TEST = test_name
    TEST_CONFIG = TEST_CONFIGS[test_name]
    CHUNK_SIZE = TEST_CONFIG["chunk_size"]
    BATCH_SIZE = TEST_CONFIG["batch_size"]
    print(f"Test config reset to: {test_name}")

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model ready — using test config: {CURRENT_TEST}")

# ============================================================
# Prepare work
# ============================================================
# --- skills ---
skills_pd = spark.sql(f"""
    SELECT DISTINCT
        skill_uuid          AS source_id,
        'skill'             AS source_type,
        job_title || ' | skill: ' || skill_label AS embed_text,
        job_title || ' | ' || skill_label         AS source_title,
        '{CURRENT_TEST}'    AS test_config
    FROM workspace.default.mcf_job_skills
    WHERE job_title IS NOT NULL AND skill_label IS NOT NULL
""").toPandas()

# --- courses ---
courses_pd = spark.sql(f"""
    SELECT
        coursereferencenumber                          AS source_id,
        'course'                                       AS source_type,
        'course: ' || coursetitle || ' | learn: ' ||
            LEFT(COALESCE(what_you_learn, ''), 1000)   AS embed_text,
        coursetitle                                    AS source_title,
        '{CURRENT_TEST}'                               AS test_config
    FROM workspace.default.my_skills_future_course_directory
    WHERE coursetitle IS NOT NULL
""").toPandas()

all_pd = pd.concat([skills_pd, courses_pd], ignore_index=True)
all_pd = all_pd.drop_duplicates(subset=["source_id"]).reset_index(drop=True)
total_candidates = len(all_pd)
print(f"Total candidates : {total_candidates:,}")

# --- subtract already-embedded rows for current test config ---
try:
    done_ids = set(
        spark.sql(f"""
            SELECT DISTINCT source_id
            FROM {CHECKPOINT_TABLE}
            WHERE test_config = '{CURRENT_TEST}'
        """).toPandas()["source_id"].tolist()
    )
    print(f"Already done     : {len(done_ids):,}")
    all_pd = all_pd[~all_pd["source_id"].isin(done_ids)].reset_index(drop=True)

    # sanity check
    expected_remaining = total_candidates - len(done_ids)
    print(f"Remaining to embed: {len(all_pd):,}")
    if len(all_pd) != expected_remaining:
        print(f"⚠️  Mismatch (expected {expected_remaining:,}) — likely duplicate source_ids across skills/courses, already deduped above")

except Exception as e:
    print(f"No checkpoint table yet — starting fresh")

print(f"Remaining to embed: {len(all_pd):,}")

In [0]:
# ============================================================
# Cell 3 — Encode in chunks, append to Delta after each chunk
# ============================================================
import time
from datetime import timedelta
from IPython.display import display, HTML
import spacy

# Download the spacy model
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

nlp = spacy.load("en_core_web_sm")

schema = StructType([
    StructField("source_type",  StringType(),           False),
    StructField("source_id",    StringType(),           False),
    StructField("source_title", StringType(),           True),
    StructField("embed_text",   StringType(),           True),
    StructField("embedding",    ArrayType(FloatType()), False),
    StructField("test_config",  StringType(),           False),
])

def update_progress(msg):
    display(HTML(f"""
        <script>
            var el = document.getElementById('embed-progress');
            if (el) el.innerHTML = `{msg}`;
        </script>
        <div id="embed-progress" style="font-family:monospace;white-space:pre">{msg}</div>
    """))

def extract_nouns(text):
    doc = nlp(text)
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "PROPN", "ADJ"]])

# -------------------------------
# Apply noun extraction by test config
# -------------------------------
course_mask = all_pd["source_type"] == "course"

if CURRENT_TEST in ["TestA", "TestA2", "TestA3"]:
    # Baseline noun extraction for courses
    all_pd.loc[course_mask, "embed_text"] = all_pd.loc[course_mask, "embed_text"].apply(extract_nouns)

elif CURRENT_TEST in ["TestB", "TestB2"]:
    # Preserve course title, extract nouns only from what_you_learn
    def extract_course_nouns(row):
        embed_text = row["embed_text"]
        parts = embed_text.split(" | learn: ")
        course_title = parts[0].replace("course: ", "") if len(parts) > 0 else ""
        what_you_learn = parts[1] if len(parts) > 1 else ""
        nouns_learn = extract_nouns(what_you_learn)
        return f"{course_title} {nouns_learn}".strip()
    all_pd.loc[course_mask, "embed_text"] = all_pd.loc[course_mask].apply(extract_course_nouns, axis=1)

# -------------------------------
# Embedding and Delta write loop
# -------------------------------
total        = len(all_pd)
chunks       = list(range(0, total, CHUNK_SIZE))
total_chunks = len(chunks)
run_start    = time.time()

for i, start in enumerate(chunks):
    chunk       = all_pd.iloc[start : start + CHUNK_SIZE].copy()
    chunk_start = time.time()

    embeddings = model.encode(
        chunk["embed_text"].tolist(),
        batch_size=BATCH_SIZE,
        show_progress_bar=False,   # off — we have our own display
        normalize_embeddings=True,
    )
    chunk["embedding"] = [e.tolist() for e in embeddings]

    df_chunk = spark.createDataFrame(
        chunk[["source_type","source_id","source_title","embed_text","embedding","test_config"]],
        schema=schema
    )

    #write_mode = "overwrite" if (i == 0 and total == len(all_pd)) else "append"
    (
        df_chunk.write
        .format("delta")
        .mode("append")
        .saveAsTable(CHECKPOINT_TABLE)
    )

    elapsed      = time.time() - run_start
    chunk_time   = time.time() - chunk_start
    rows_per_sec = len(chunk) / chunk_time
    rows_done    = start + len(chunk)
    pct          = rows_done / total * 100
    bar_filled   = int(pct / 2)
    bar          = "█" * bar_filled + "░" * (50 - bar_filled)
    
    remaining    = total - rows_done
    eta_sec      = remaining / rows_per_sec if rows_per_sec > 0 else 0

    msg = (
        f"[{bar}] {pct:.1f}%\n"
        f"Chunk  : {i+1:,} / {total_chunks:,}\n"
        f"Rows   : {rows_done:,} / {total:,}\n"
        f"Speed  : {rows_per_sec:.0f} rows/sec\n"
        f"Elapsed: {str(timedelta(seconds=int(elapsed)))}\n"
        f"ETA    : {str(timedelta(seconds=int(eta_sec)))}"
    )
    update_progress(msg)

update_progress(f"✅ ALL DONE — {total:,} rows in {str(timedelta(seconds=int(time.time()-run_start)))}")
display(spark.sql(f"SELECT source_type, COUNT(*) n FROM {CHECKPOINT_TABLE} GROUP BY 1"))

In [0]:
# Cell 4 — Batch test: cycle through skills and export to CSV with cosine similarity and LLM to check.
# ============================================================
from datetime import datetime
import pandas as pd
import numpy as np

test_skills_list = [
    {"skill": "Python", "frequency": 8},
    {"skill": "SQL", "frequency": 5},
    {"skill": "C++", "frequency": 4},
    {"skill": "Compliance", "frequency": 3},
    {"skill": "CI/CD", "frequency": 3},
    {"skill": "Kubernetes", "frequency": 3},
    {"skill": "Strategy", "frequency": 2},
    {"skill": "SDN", "frequency": 2},
    {"skill": "Incident Management", "frequency": 2},
    {"skill": "Virtualization", "frequency": 2},
    {"skill": "User Research", "frequency": 2},
    {"skill": "React", "frequency": 2},
    {"skill": "SIEM", "frequency": 2},
    {"skill": "Scripting", "frequency": 2},
    {"skill": "Microservices", "frequency": 2},
    {"skill": "Cloud Billing", "frequency": 1},
    {"skill": "VMware", "frequency": 1},
    {"skill": "Reporting", "frequency": 1},
    {"skill": "Juniper", "frequency": 1},
    {"skill": "Tagging Strategy", "frequency": 1},
    {"skill": "Routing/Switching", "frequency": 1},
    {"skill": "Wireless", "frequency": 1},
    {"skill": "Cost Optimization", "frequency": 1},
    {"skill": "Shell", "frequency": 1},
    {"skill": "Hardware", "frequency": 1},
    {"skill": "Linux Server", "frequency": 1},
    {"skill": "Windows Server", "frequency": 1},
    {"skill": "Networking", "frequency": 1},
    {"skill": "Storage", "frequency": 1},
    {"skill": "Cisco", "frequency": 1},
    {"skill": "Active Directory", "frequency": 1},
    {"skill": "Hybrid Cloud", "frequency": 1},
    {"skill": "Database Migration", "frequency": 1},
    {"skill": "KVM", "frequency": 1},
    {"skill": "Container Security", "frequency": 1},
    {"skill": "Authentication", "frequency": 1},
    {"skill": "GRC", "frequency": 1},
    {"skill": "Quantitative Risk", "frequency": 1},
    {"skill": "Threat Modeling", "frequency": 1},
    {"skill": "Log Analysis", "frequency": 1},
    {"skill": "Incident Categorization", "frequency": 1},
    {"skill": "Communication", "frequency": 1},
    {"skill": "Security Automation", "frequency": 1},
    {"skill": "SAST/DAST", "frequency": 1},
    {"skill": "AWS", "frequency": 1},
    {"skill": "Refactoring", "frequency": 1},
    {"skill": "Azure", "frequency": 1},
    {"skill": "GCP", "frequency": 1},
    {"skill": "Solution Architecture", "frequency": 1},
    {"skill": "Cost Management", "frequency": 1},
    {"skill": "Observability", "frequency": 1},
    {"skill": "Automation", "frequency": 1},
    {"skill": "Internal Developer Platforms", "frequency": 1},
    {"skill": "Infrastructure as Code", "frequency": 1},
    {"skill": "Lift and Shift", "frequency": 1},
    {"skill": "Hyper-V", "frequency": 1},
    {"skill": "Rack Management", "frequency": 1},
    {"skill": "Storage Networking", "frequency": 1},
    {"skill": "Finance Logic", "frequency": 1},
    {"skill": "Prototyping", "frequency": 1},
    {"skill": "Change Management", "frequency": 1},
    {"skill": "Stakeholder Influence", "frequency": 1},
    {"skill": "ROI Analysis", "frequency": 1},
    {"skill": "ITIL", "frequency": 1},
    {"skill": "SLA Management", "frequency": 1},
    {"skill": "Problem Management", "frequency": 1},
    {"skill": "Vendor Mgmt", "frequency": 1},
    {"skill": "SAP S/4HANA", "frequency": 1},
    {"skill": "Supply Chain Logic", "frequency": 1},
    {"skill": "Wireframing", "frequency": 1},
    {"skill": "Configuration", "frequency": 1},
    {"skill": "Technical Presentations", "frequency": 1},
    {"skill": "Proof of Concept", "frequency": 1},
    {"skill": "Product Demo", "frequency": 1},
    {"skill": "RFP", "frequency": 1},
    {"skill": "Solidity", "frequency": 1},
    {"skill": "Smart Contracts", "frequency": 1},
    {"skill": "Ethereum", "frequency": 1},
    {"skill": "Cryptography", "frequency": 1},
    {"skill": "Accessibility", "frequency": 1},
    {"skill": "Figma", "frequency": 1},
    {"skill": "Cooling", "frequency": 1},
    {"skill": "Requirements", "frequency": 1},
    {"skill": "Power Systems", "frequency": 1},
    {"skill": "OAuth", "frequency": 1},
    {"skill": "Physical Security", "frequency": 1},
    {"skill": "Agile", "frequency": 1},
    {"skill": "Scrum", "frequency": 1},
    {"skill": "Stakeholder Management", "frequency": 1},
    {"skill": "JIRA", "frequency": 1},
    {"skill": "Budgeting", "frequency": 1},
    {"skill": "Roadmapping", "frequency": 1},
    {"skill": "API Specs", "frequency": 1},
    {"skill": "Integration", "frequency": 1},
    {"skill": "Technical Specs", "frequency": 1},
    {"skill": "Requirements Gathering", "frequency": 1},
    {"skill": "Process Mapping", "frequency": 1},
    {"skill": "User Stories", "frequency": 1},
    {"skill": "ERP", "frequency": 1},
    {"skill": "CRM", "frequency": 1},
    {"skill": "SAP", "frequency": 1},
    {"skill": "Salesforce", "frequency": 1},
    {"skill": "Business Process", "frequency": 1},
    {"skill": "SAML", "frequency": 1},
    {"skill": "ISO 27001", "frequency": 1},
    {"skill": "Azure AD", "frequency": 1},
    {"skill": "Tokenization", "frequency": 1},
    {"skill": "Data Storytelling", "frequency": 1},
    {"skill": "Data Privacy", "frequency": 1},
    {"skill": "Risk Assessment", "frequency": 1},
    {"skill": "Policy Writing", "frequency": 1},
    {"skill": "OpenCV", "frequency": 1},
    {"skill": "Deep Learning", "frequency": 1},
    {"skill": "Image Processing", "frequency": 1},
    {"skill": "Transformers", "frequency": 1},
    {"skill": "BERT/GPT", "frequency": 1},
    {"skill": "Linguistics", "frequency": 1},
    {"skill": "Excel", "frequency": 1},
    {"skill": "Java", "frequency": 1},
    {"skill": "Power BI", "frequency": 1},
    {"skill": "JavaScript", "frequency": 1},
    {"skill": "Tableau", "frequency": 1},
    {"skill": "Node.js", "frequency": 1},
    {"skill": "MongoDB", "frequency": 1},
    {"skill": "HTML/CSS", "frequency": 1},
    {"skill": "Go", "frequency": 1},
    {"skill": "API Design", "frequency": 1},
    {"skill": "Vue", "frequency": 1},
    {"skill": "TypeScript", "frequency": 1},
    {"skill": "Web Performance", "frequency": 1},
    {"skill": "Swift", "frequency": 1},
    {"skill": "Statistics", "frequency": 1},
    {"skill": "R", "frequency": 1},
    {"skill": "Kotlin", "frequency": 1},
    {"skill": "Flutter", "frequency": 1},
    {"skill": "React Native", "frequency": 1},
    {"skill": "Mobile UX", "frequency": 1},
    {"skill": "Unity", "frequency": 1},
    {"skill": "Unreal Engine", "frequency": 1},
    {"skill": "C#", "frequency": 1},
    {"skill": "3D Mathematics", "frequency": 1},
    {"skill": "C", "frequency": 1},
    {"skill": "RTOS", "frequency": 1},
    {"skill": "PyTorch", "frequency": 1},
    {"skill": "TensorFlow", "frequency": 1},
    {"skill": "Microcontrollers", "frequency": 1},
    {"skill": "Firmware", "frequency": 1},
    {"skill": "Multithreading", "frequency": 1},
    {"skill": "MLOps", "frequency": 1},
    {"skill": "Low Latency", "frequency": 1},
    {"skill": "Systems Programming", "frequency": 1},
    {"skill": "Distributed Systems", "frequency": 1},
    {"skill": "Security", "frequency": 1},
    {"skill": "Design Patterns", "frequency": 1},
    {"skill": "Generative AI", "frequency": 1},
    {"skill": "Selenium", "frequency": 1},
    {"skill": "Cypress", "frequency": 1},
    {"skill": "Jenkins", "frequency": 1},
    {"skill": "Test Planning", "frequency": 1},
    {"skill": "Terraform", "frequency": 1},
    {"skill": "LLMs", "frequency": 1},
    {"skill": "RAG", "frequency": 1},
    {"skill": "Prompt Engineering", "frequency": 1},
    {"skill": "Ansible", "frequency": 1},
    {"skill": "Network Security", "frequency": 1},
    {"skill": "Firewalls", "frequency": 1},
    {"skill": "LangChain", "frequency": 1},
    {"skill": "Threat Intelligence", "frequency": 1},
    {"skill": "Security Frameworks", "frequency": 1},
    {"skill": "Vector Databases", "frequency": 1},
    {"skill": "Zero Trust", "frequency": 1},
    {"skill": "Spark", "frequency": 1},
    {"skill": "Risk Management", "frequency": 1},
    {"skill": "IAM", "frequency": 1},
    {"skill": "Forensics", "frequency": 1},
    {"skill": "Kafka", "frequency": 1},
    {"skill": "Threat Hunting", "frequency": 1},
    {"skill": "Malware Analysis", "frequency": 1},
    {"skill": "Airflow", "frequency": 1},
    {"skill": "AWS Security", "frequency": 1},
    {"skill": "Azure Security", "frequency": 1},
    {"skill": "ETL Pipelines", "frequency": 1},
    {"skill": "Identity Management", "frequency": 1},
    {"skill": "Encryption", "frequency": 1},
    {"skill": "Metasploit", "frequency": 1},
    {"skill": "System Design", "frequency": 1},
    {"skill": "Burp Suite", "frequency": 1},
    {"skill": "Network Auditing", "frequency": 1},
    {"skill": "Cloud AI Services", "frequency": 1},
    {"skill": "Vulnerability Assessment", "frequency": 1},
    {"skill": "MAS TRM Guidelines", "frequency": 1},
    {"skill": "API Integration", "frequency": 1},
    {"skill": "Scalability", "frequency": 1},
    {"skill": "PDPA", "frequency": 1},
    {"skill": "Audit", "frequency": 1},
    {"skill": "Okta", "frequency": 1},
    {"skill": "Docker", "frequency": 1},
    {"skill": "Model Monitoring", "frequency": 1},
    {"skill": "Cloud Computing", "frequency": 1},
    {"skill": "Algorithms", "frequency": 1},
    {"skill": "Web3.js", "frequency": 1},
    {"skill": "AI Frameworks", "frequency": 1},
    {"skill": "Reasoning Systems", "frequency": 1},
    {"skill": "Random Forest", "frequency": 1},
    {"skill": "Search algorithms", "frequency": 1},
    {"skill": "Agentic systems", "frequency": 1},
]

TOP_K           = 10
TOP_N           = 100
MIN_SIMILARITY  = 0.30
FILTER_TYPE     = "course"  # "course", "skill", or None

# Load embeddings once (filter by current test config)
filter_clause = f"WHERE test_config = '{CURRENT_TEST}'"
if FILTER_TYPE:
    filter_clause += f" AND source_type = '{FILTER_TYPE}'"

print(f"Loading embeddings from {CHECKPOINT_TABLE}...")
all_embeddings = spark.sql(f"""
    SELECT source_type, source_id, source_title, embedding
    FROM {CHECKPOINT_TABLE}
    {filter_clause}
""").toPandas()
emb_matrix = np.vstack(all_embeddings["embedding"].values)
print(f"Loaded {len(all_embeddings):,} embeddings")

# Process each skill
all_results = []
total_skills = len(test_skills_list)

for idx, skill_data in enumerate(test_skills_list, 1):
    skill_name = skill_data["skill"]
    frequency = skill_data["frequency"]
    
    # Encode query
    query_vec = model.encode([skill_name], normalize_embeddings=True)[0]
    
    # Compute similarity scores
    scores = emb_matrix @ query_vec
    results_df = all_embeddings.copy()
    results_df["score"] = scores
    
    # Sort and filter
    top_results = results_df.sort_values("score", ascending=False).head(TOP_N)
    display_results = top_results[top_results["score"] >= MIN_SIMILARITY].head(TOP_K)
    if len(display_results) < TOP_K:
        display_results = top_results.head(TOP_K)
    
    # Add metadata
    for rank, (_, row) in enumerate(display_results.iterrows(), 1):
        all_results.append({
            "query_skill": skill_name,
            "frequency": frequency,
            "rank": rank,
            "source_type": row["source_type"],
            "source_id": row["source_id"],
            "source_title": row["source_title"],
            "score": row["score"]
        })
    
    if idx % 10 == 0 or idx == total_skills:
        print(f"Processed {idx}/{total_skills} skills...")

# Create final DataFrame
results_df = pd.DataFrame(all_results)

# Generate filename with test config and timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"skill_search_results_{CURRENT_TEST}_{timestamp}.csv"

# Save to CSV
results_df.to_csv(filename, index=False)
print(f"\n✅ Results saved to: {filename}")
print(f"Total rows: {len(results_df):,}")
print(f"Total unique skills: {results_df['query_skill'].nunique()}")

# Display summary
display(results_df.head(20))

In [0]:
# ASK ChatGPT to gauge the course quality based on a set rubric


import pandas as pd
import requests
import time

OPENAI_API_KEY = None  # <-- Insert your OpenAI key here
MODEL = "gpt-4o-mini"  # gpt4mini

def batch_rows(df, batch_size=200):
    for i in range(0, len(df), batch_size):
        yield df.iloc[i:i+batch_size]

def build_prompt(batch):
    rubric = """
You are a relevance assessor in an academic evaluation study. You will receive rows containing a target skill and a recommended course title. Score each pair using the rubric below.
SCORES
0 = Irrelevant. The course has no connection to the skill.
1 = Tangential. Same broad domain or a prerequisite, but does not directly teach the skill.
2 = Partial. The skill is covered but is not the main focus of the course.
3 = Directly relevant. The course is primarily about the skill.
RULES

Judge the title as written. Do not use outside knowledge.
Rate each pair independently.
When uncertain between two scores, pick the higher one.
Prerequisites are at most a 1.

Rows:
"""
    rows = ""
    for idx, row in batch.iterrows():
        rows += f"{idx+1}. Skill: {row['query_skill']} | Course: {row['source_title']}\n"
    prompt = rubric + rows + "\nReturn your scores as a numbered list, e.g.:\n1: 2\n2: 1\n..."
    return prompt

def call_openai(prompt):
    url = "https://api.openai.com/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {OPENAI_API_KEY}",
        "Content-Type": "application/json"
    }
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0
    }
    response = requests.post(url, headers=headers, json=payload)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

def parse_scores(response, batch):
    scores = {}
    for line in response.strip().splitlines():
        if ":" in line:
            idx, score = line.split(":")
            idx = int(idx.strip())
            score = int(score.strip())
            scores[idx] = score
    batch_scores = []
    for i, row in enumerate(batch.itertuples(), 1):
        batch_scores.append(score if (score := scores.get(i)) is not None else None)
    return batch_scores

def process_test_config(input_path, output_path, test_name):
    print(f"\n{'='*60}")
    print(f"Processing {test_name}")
    print(f"Input: {input_path}")
    print(f"Output: {output_path}")
    print(f"{'='*60}")
    
    df = pd.read_csv(input_path)
    print(f"Loaded {len(df):,} rows")
    
    results = []
    total_batches = (len(df) + 199) // 200  # ceiling division
    
    for batch_num, batch in enumerate(batch_rows(df, batch_size=200), 1):
        print(f"  Processing batch {batch_num}/{total_batches}...", end="")
        prompt = build_prompt(batch)
        try:
            response = call_openai(prompt)
            batch_scores = parse_scores(response, batch)
            batch_result = batch.copy()
            batch_result["openai_score"] = batch_scores
            results.append(batch_result)
            print(" ✓")
        except Exception as e:
            print(f" ✗ Error: {e}")
            time.sleep(10)
            continue
        time.sleep(2)  # avoid rate limits
    
    final_df = pd.concat(results, ignore_index=True)
    final_df.to_csv(output_path, index=False)
    print(f"\n✅ {test_name} complete: {len(final_df):,} rows saved to {output_path}")
    return final_df

# --- Main: Process each test separately ---
#test_a_path = "skill_search_results_TestA_20260427_112151.csv"
#test_b_path = "skill_search_results_TestB_20260430_155736.csv"
test_a3_path = "skill_search_results_TestA3_20260501_035556.csv"

judged_a3 = process_test_config(test_a3_path, "skill_course_judgedA3.csv", "TestA3")

print(f"\n{'='*60}")
print("ALL PROCESSING COMPLETE")
print(f"{'='*60}")
print(f"TestA3: {len(judged_a3):,} rows → skill_course_judgedA3.csv")
print(f"Total: {len(judged_a3):,} rows judged")

In [0]:
# Download 10 course title examples to see SpaCY effect on noun extraction: raw vs noun-extracted
# ============================================================
import spacy
import pandas as pd

# Download and load spacy model if not already loaded
try:
    nlp
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

def extract_nouns(text):
    doc = nlp(text)
    return " ".join([token.text for token in doc if token.pos_ in ["NOUN", "PROPN", "ADJ"]])

# Get 10 course examples from the source table
courses_df = spark.sql("""
    SELECT
        coursereferencenumber AS source_id,
        coursetitle,
        COALESCE(what_you_learn, '') AS what_you_learn
    FROM workspace.default.my_skills_future_course_directory
    WHERE coursetitle IS NOT NULL
    LIMIT 10
""").toPandas()

# Compose raw embed_text and noun-extracted version
courses_df["raw_embed_text"] = "course: " + courses_df["coursetitle"] + " | learn: " + courses_df["what_you_learn"].str[:1000]
courses_df["noun_embed_text"] = courses_df["raw_embed_text"].apply(extract_nouns)
courses_df["coursetitle_nouns"] = courses_df["coursetitle"].apply(extract_nouns)

# Select relevant columns for display
examples_df = courses_df[["coursetitle", "coursetitle_nouns", "raw_embed_text", "noun_embed_text"]]

display(examples_df)

In [0]:
# Visualize the data in a 2D plot for skill Python as an example to see cosine proximity
import json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

TOP_N = 10
SKILL_QUERY = "Python"

def get_neighbors(test_config, skill_query, top_n):
    df = spark.sql(f"""
        SELECT source_type, source_id, source_title, embedding
        FROM {CHECKPOINT_TABLE}
        WHERE test_config = '{test_config}'
    """).toPandas()

    df["embedding"] = df["embedding"].apply(lambda x: np.array(x, dtype=np.float32))

    emb_matrix = np.vstack(df["embedding"].values)

    model = SentenceTransformer("all-MiniLM-L6-v2")
    query_vec = model.encode([skill_query], normalize_embeddings=True)[0]

    norms = np.linalg.norm(emb_matrix, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1, norms)
    emb_matrix_normed = emb_matrix / norms

    scores = emb_matrix_normed @ query_vec
    df = df.copy()
    df["score"] = scores

    # Take top_n from each type separately
    top_courses = df[df["source_type"] == "course"].sort_values("score", ascending=False).head(top_n)
    top_skills  = df[df["source_type"] == "skill"].sort_values("score", ascending=False).head(top_n)
    top_df = pd.concat([top_courses, top_skills]).reset_index(drop=True)

    return top_df, query_vec


neighbors_a3, query_vec_a3 = get_neighbors("TestC", SKILL_QUERY, TOP_N)

# Stack all embeddings for t-SNE
all_embeds = np.vstack([
    np.vstack(neighbors_a3["embedding"].values),
    query_vec_a3.reshape(1, -1),
])

tsne = TSNE(n_components=2, random_state=42, perplexity=5, n_iter=1000, init="pca")
xy = tsne.fit_transform(all_embeds)

n_a3 = len(neighbors_a3)
xy_a3       = xy[:n_a3]
xy_query_a3 = xy[-1]

# Split by source_type
courses_mask = neighbors_a3["source_type"] == "course"
skills_mask  = neighbors_a3["source_type"] == "skill"

fig, ax = plt.subplots(figsize=(11, 7))

# Plot courses as steelblue
ax.scatter(xy_a3[courses_mask, 0], xy_a3[courses_mask, 1],
           c="steelblue", label="Courses", s=60, zorder=3)

# Plot skills as orange
ax.scatter(xy_a3[skills_mask, 0], xy_a3[skills_mask, 1],
           c="orange", label="Skills", s=60, zorder=3)

# Plot query star
ax.scatter(*xy_query_a3, c="navy", marker="*", s=250,
           label=f'"{SKILL_QUERY}" query', zorder=4)

# Labels
for idx in neighbors_a3[courses_mask].index:
    ax.text(xy_a3[idx, 0], xy_a3[idx, 1], neighbors_a3.loc[idx, "source_title"],
            fontsize=7.5, color="steelblue", va="bottom")

for idx in neighbors_a3[skills_mask].index:
    ax.text(xy_a3[idx, 0], xy_a3[idx, 1], neighbors_a3.loc[idx, "source_title"],
            fontsize=7.5, color="darkorange", va="bottom")

ax.set_title(f'Nearest Neighbors to "{SKILL_QUERY}": TestC')
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()